In [4]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import matplotlib
import matplotlib.pyplot as plt
# Use the "Agg" backend for matplotlib to avoid issues
matplotlib.use("Agg")

import json
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

from modules.dataset import ICBHIAudioDataset, KAUHAudioDataset
from modules.lungsound import LungSoundAudio
from modules.transforms import *

In [16]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
RAW_DATA_FOLDER = DATA_PATH / "data_raw"
PREPROCESSED_DATA_FOLDER = DATA_PATH / "data_preprocessed_new"

if not os.path.exists(RAW_DATA_FOLDER):
    raise FileNotFoundError(f"Raw data folder not found at {RAW_DATA_FOLDER}. Please ensure the original data was already downloaded and placed in the correct location.")

if not os.path.exists(PREPROCESSED_DATA_FOLDER):
    os.makedirs(PREPROCESSED_DATA_FOLDER)
else:
    if len(os.listdir(PREPROCESSED_DATA_FOLDER)) > 0:
        print(f"[WARNING] Preprocessed data folder already exist and is not empty ({PREPROCESSED_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

[WARNING] Preprocessed data folder already exist and is not empty (/home/leticialopes/projects/IA901/IA901_Project/data/data_preprocessed_new). Consider deleting it to run the preprocessing step again.


In [ ]:
TARGET_SR = 22050   # Hz
WINDOW_LENGTH = 5.0 # seconds
HOP_LENGTH = 5.0    # seconds


def save_features_as_npy(features: np.ndarray, path: str):
    """
    Saves the extracted features to a .npy file.
    Args:
        features (np.ndarray): The extracted features to be saved.
        path (str): The path where the features will be saved.
    """
    np.save(path, features)


def save_features_as_png(features: np.ndarray, path: str, spec_params: dict = None):
    """
    Saves the extracted features as a PNG image.
    Args:
        features (np.ndarray): The extracted features to be saved.
        path (str): The path where the features will be saved.
        spec_params (dict): Parameters for the spectrogram visualization.
    """
    params = {} if spec_params is None else spec_params

    fig, ax = plt.subplots()
    librosa.display.specshow(features, ax=ax, **params)
    ax.axis("off")
    fig.savefig(path, bbox_inches="tight", pad_inches=0)
    ax.clear()
    fig.clf()
    plt.close(fig)


def preprocess_and_save(original_data_path: Path, preprocessed_data_path: Path, save_as: str = "npy"):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location using multiple feature extractors.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
        save_as (str): Format to save the preprocessed data ('npy', 'npz', or 'png').
    """
    datasets = [
        ICBHIAudioDataset(original_data_path),
        KAUHAudioDataset(original_data_path)
    ]
    feature_extractors = [
        STFT(),
        MelSpectrogram(n_mels=128),
        MFCC(n_mfcc=128),
        MFCCDelta(n_mfcc=128),
        Chroma(n_chroma=128),
        SpectralContrast(n_bands=4, fmin=50),
        CQT(n_bins=48, fmin=30, bins_per_octave=12),
        Phase(),
    ]

    # Iterate through each dataset and apply preprocessing with each feature extractor
    for dataset in datasets:
        df = dataset.data
        new_rows = []
        computed_data = False
        for feature_extractor in feature_extractors:
            for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {dataset.name} with {feature_extractor.name}"):
                audio_path = row["FilePath"]
                audio = LungSoundAudio(audio_path)

                # Apply preprocessing transforms
                # 1. Split the audio into windows of fixed duration
                cropped_audios = Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH)(audio)
                for i, cropped_audio in enumerate(cropped_audios):
                    # # 2. Resample the audio to the target sampling rate
                    # resampled_audio = Resample(target_sr=TARGET_SR)(cropped_audio)
                    # # 3. Normalize the audio to have zero mean and unit variance
                    # normalized_audio = NormalizeAudio()(resampled_audio)
                    # # 4. Extract features using the provided feature extractor
                    # features = feature_extractor(normalized_audio)

                    # Save the preprocessed audio to the new location
                    diagnosis = str(row["Diagnosis"])
                    start = int(i * HOP_LENGTH)
                    end = int(start + WINDOW_LENGTH)
                    new_file_name = f"{Path(audio_path).stem}_clip-{start:03d}-{end:03d}.{save_as}"
                    preprocessed_file_path = preprocessed_data_path / dataset.name / save_as / feature_extractor.name / diagnosis / new_file_name
                    preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)

                    # if save_as == "npy":
                    #     save_features_as_npy(features, preprocessed_file_path)
                    # elif save_as == "png":
                    #     params = feature_extractor.plot_params
                    #     params["sr"] = TARGET_SR
                    #     save_features_as_png(features, preprocessed_file_path, params)
                    # else:
                    #     raise ValueError(f"Unsupported save format: {save_as}")

                    # Add the row to the new df
                    if not computed_data:
                        new_row = row.copy()
                        new_row["FilePath"] = preprocessed_file_path.stem
                        new_rows.append(new_row)

            computed_data = True
            # Save the preprocessing parameters to a json file
            params_path = preprocessed_data_path / dataset.name / save_as / feature_extractor.name / "preprocessing.json"
            params = {
                Window.__name__: vars(Window(window_length=WINDOW_LENGTH, hop_length=HOP_LENGTH)),
                Resample.__name__: vars(Resample(target_sr=TARGET_SR)),
                NormalizeAudio.__name__: vars(NormalizeAudio()),
                feature_extractor.__class__.__name__: vars(feature_extractor)
            }
            with open(params_path, "w") as f:
                json.dump(params, f, indent=4)

        # Save the new data to a CSV file
        data_path = preprocessed_data_path / dataset.name / "data.csv"
        new_df = pd.DataFrame(new_rows).rename(columns={"FilePath": "FileName"})
        new_df.to_csv(data_path, index=False)

        print(f"Preprocessing for {dataset.name} completed. Total preprocessed audio files: {len(new_rows)}")
        print(f"Data saved to {os.path.relpath(data_path, start=os.getcwd())}")

In [ ]:
preprocess_and_save(RAW_DATA_FOLDER, PREPROCESSED_DATA_FOLDER, save_as="npy")
# 6m

In [ ]:
preprocess_and_save(RAW_DATA_FOLDER, PREPROCESSED_DATA_FOLDER, save_as="png")
# 35m

In [ ]:
from PIL import Image

img = Image.open('../data/data_preprocessed/ICBHI_2017/png/chroma/Asthma/103_2b2_Ar_mc_LittC2SE_clip-000-005.png')

print(img.size, img.mode, img.format)

(496, 369) RGBA PNG ('R', 'G', 'B', 'A')
